In [158]:
# https://platform.olimpiada-ai.ro/ro/problems/186

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [159]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/annual-income-prediction/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/annual-income-prediction/test.csv")

train['income'] = train['income'].map(dict([('<=50K', 0), ('>50K', 1)]))

train.shape, test.shape

((39073, 17), (9769, 16))

In [160]:
train['income'].value_counts(normalize=True)

income
0    0.76073
1    0.23927
Name: proportion, dtype: float64

In [161]:
pd.concat([train.isna().sum(), test.isna().sum()], axis=1, keys=['train', 'test'])

,train,test
sampleid,0,0.0
age,0,0.0
workclass,2224,575.0
fnlwgt,0,0.0
education,0,0.0
educational_num,0,0.0
marital_status,0,0.0
occupation,2234,575.0
relationship,0,0.0
race,0,0.0


In [162]:
train.head()

,sampleid,age,workclass,fnlwgt,education,educational_num,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,native_country,income,profile_description
0,34343,71,Private,77253,HS-grad,9,Never-married,Handlers-cleaners,Not-in-family,White,Male,0,0,17,United-States,0,welding analysis operations timber crane certi...
1,18560,17,Private,329783,10th,6,Never-married,Sales,Other-relative,White,Female,0,0,10,United-States,0,merger KPI leverage management certified profe...
2,12478,27,Private,91257,HS-grad,9,Married-civ-spouse,Other-service,Husband,White,Male,0,0,40,El-Salvador,0,support garnish turndown experience concierge
3,561,43,Private,125577,HS-grad,9,Separated,Adm-clerical,Unmarried,Black,Female,0,0,40,United-States,0,floral amenity concierge linen training operat...
4,3428,31,Private,137978,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,United-States,0,margin monitoring merger maintenance portfolio


In [163]:
country_ans = train.groupby('native_country')['income'].sum().sort_values(ascending=False).index[1]
country_ans

'Philippines'

In [164]:
occ_ans = train.groupby('occupation')['income'].mean().sort_values(ascending=False).index[0]
occ_ans

'Exec-managerial'

In [165]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

train[cat_features] = imputer.fit_transform(train[cat_features])
test[cat_features] = imputer.transform(test[cat_features])

In [166]:
from sklearn.model_selection import train_test_split

id_col = 'sampleid'
target_col = 'income'
features = [c for c in train.columns if c not in [id_col, target_col]]
text_features = ['profile_description']
cat_features = [c for c in train.select_dtypes('object').columns if c in features and c not in text_features]

X, y = train[features], train[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.1, random_state=42)
X_train.shape, X_valid.shape

((35165, 15), (3908, 15))

In [167]:
from catboost import Pool

train_pool = Pool(X_train, y_train, cat_features=cat_features, text_features=text_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features, text_features=text_features)

In [168]:
from catboost import CatBoostClassifier

params = {
    'iterations': 600,
    'loss_function': 'Logloss',
    'eval_metric': 'TotalF1:average=Macro',
    'metric_period': 100,
    'random_state': 42,
    'max_depth': 2
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

Learning rate set to 0.095346
0:	learn: 0.7051557	test: 0.7040717	best: 0.7040717 (0)	total: 41.6ms	remaining: 24.9s
100:	learn: 0.7877071	test: 0.8001779	best: 0.8001779 (100)	total: 3.47s	remaining: 17.1s
200:	learn: 0.7990289	test: 0.8108449	best: 0.8108449 (200)	total: 6.68s	remaining: 13.3s
300:	learn: 0.8026010	test: 0.8130466	best: 0.8130466 (300)	total: 9.8s	remaining: 9.74s
400:	learn: 0.8048537	test: 0.8138402	best: 0.8138402 (400)	total: 12.9s	remaining: 6.42s
500:	learn: 0.8071199	test: 0.8153923	best: 0.8153923 (500)	total: 16s	remaining: 3.16s
599:	learn: 0.8090518	test: 0.8151143	best: 0.8153923 (500)	total: 19s	remaining: 0us

bestTest = 0.8153923033
bestIteration = 500

Shrink model to first 501 iterations.


In [169]:
y_pred = model.predict(test[features])
y_pred = np.array(list(map(dict([(0, '<=50K'), (1, '>50K')]).get, y_pred)))

In [170]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 1))

X_test_vectorized = vectorizer.fit_transform(test['profile_description']).toarray()
print('Vectorized shape:', X_test_vectorized.shape)

scaler = StandardScaler()

X_test_vectorized = scaler.fit_transform(X_test_vectorized)

clusterer = KMeans(n_clusters=4)

clusters = clusterer.fit_predict(X_test_vectorized)

pd.Series(clusters).value_counts().sort_index()

Vectorized shape: (9769, 138)


0    1333
1    2940
2    2644
3    2852
Name: count, dtype: int64

In [171]:
subm = pd.DataFrame({
    'subtaskID': [1, 2] + [3] * len(test) + [4] * len(test),
    'datapointID': [1, 2] + test[id_col].tolist() + test[id_col].tolist(),
    'answer': [country_ans, occ_ans] + y_pred.tolist() + clusters.tolist()
})

subm.to_csv("submission.csv", index=False)

subm

,subtaskID,datapointID,answer
0,1,1,Philippines
1,2,2,Exec-managerial
2,3,40343,<=50K
3,3,47681,<=50K
4,3,525,>50K
...,...,...,...
19535,4,20385,3
19536,4,28001,1
19537,4,22164,1
19538,4,24734,2
